In [1]:
!pip install transformers pdf2image pdfminer.six


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.8 MB/s eta 0:00:00


In [2]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from pdfminer.high_level import extract_text
from pdf2image import convert_from_path
import requests

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

Saving 68 sidor.pdf to 68 sidor.pdf


In [5]:
!apt-get install -y poppler-utils
pages = convert_from_path("/content/68 sidor.pdf")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (216 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
# Ladda TrOCr
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-stage1")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-stage1")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-stage1 and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
ocr_texts = []
for page in pages:
    pixel_values = processor(images=page, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values)
    ocr_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ocr_texts.append(ocr_text)

In [8]:
# 4. Spara som Markdown
with open("output.md", "w", encoding="utf-8") as f:
    f.write("# Extraherad textlager\n\n")
    f.write(text + "\n\n")
    for i, ocr in enumerate(ocr_texts, 1):
        f.write(f"# OCR sida {i}\n\n{ocr}\n\n")

In [9]:
# Läs in och visa innehållet i output.md
with open("output.md", "r", encoding="utf-8") as f:
    content = f.read()

print(content[:10000])  # visa de första 10000 tecknen


# Extraherad textlager

Årsredovisning 
2022

Det här är 
FlexQube

FlexQube är en global leverantör av modulära 
och robusta mekaniska vagnar och robotiserade 
lösningar för materialhantering. Koncernen 
grundades 2010 och har sedan dess säkrat ett stort 
antal framstående företag som kunder.

FlexQube är ett teknikbolag med huvudkontor i 
Göteborg samt egna verksamheter i USA, Mexiko, 
Tyskland och England. Bolaget är verksamt inom 
vagnsbaserad materialhantering genom ett 
patenterat modulkoncept. FlexQube utvecklar 
och designar kundanpassade lösningar för både 
robotiserad och mekaniserad vagnslogistik. 
Genom företagets egenutvecklade och unika 
automationskoncept erbjuds robusta och 
självkörande robotvagnar. FlexQube har över 1000 
kunder i 37 länder där de primära marknaderna är 
Nordamerika och Europa.

FlexQubes kunder återfinns inom bland 
annat tillverkningsindustrin, distribution- och 
lagerverksamheter. Exempel på större kunder 
är Tesla, Amazon, Volvo Cars, Siemens, Au

In [10]:
import json
with open("extraction_result.json", "w", encoding="utf-8") as f:
    json.dump({"textlayer": text, "ocr_pages": ocr_texts}, f, indent=2, ensure_ascii=False)
